# Labs · Workshop SageMaker AI

Ejecuta las celdas **en orden** en el mismo kernel: cada lab reutiliza variables del anterior.

Antes de empezar:
1. Rellena `config.py` (rol de ejecución, bucket, región).
2. Por defecto `USE_SYNTHETIC = True`: se generan sesiones de carga ficticias y no necesitas ninguna tabla.
3. Cuenta dev o personal. **No te saltes el Lab 7 (limpieza)**: un endpoint olvidado sigue cobrando.

La teoría de cada lab está en el `README.md` de la carpeta superior.

In [ ]:
import time
import awswrangler as wr
import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError
from sagemaker.core import image_uris  # SDK v2: from sagemaker import image_uris

from config import (
    ATHENA_DATABASE, BASE, BUCKET, DATASET, REGION, ROLE_ARN, TS, USE_SYNTHETIC,
    aas, s3, sm, smr,
)

print(f"Región: {REGION} | datos en s3://{BUCKET}/{BASE} | sufijo de esta ejecución: {TS}")
print(boto3.client("sts").get_caller_identity()["Account"])

## Lab 1 · Dataset ML

Objetivo: split temporal, CSV para XGBoost built-in (target primero, sin cabecera) y Parquet para script mode.

In [ ]:
TARGET = "energy_kwh"
FEATURES = ["hour", "dow", "charger_max_power_kw", "connector"]


def synthetic_sessions(n: int = 20_000, seed: int = 42) -> pd.DataFrame:
    """Sesiones de carga ficticias con una relación razonable entre features y kWh."""
    rng = np.random.default_rng(seed)
    start_ts = pd.Timestamp("2025-10-01") + pd.to_timedelta(rng.integers(0, 365 * 24 * 60, n), unit="m")
    hour = start_ts.hour.to_numpy()
    dow = start_ts.dayofweek.to_numpy()
    power = rng.choice([22, 50, 150, 300], size=n, p=[0.3, 0.3, 0.3, 0.1])
    connector = np.where(power == 22, 2, rng.choice([0, 1], size=n, p=[0.85, 0.15]))
    peak = np.isin(hour, [8, 9, 18, 19])
    energy = np.minimum(power, 120) * rng.uniform(0.15, 0.6, n) * (1 + 0.2 * peak) + rng.normal(0, 3, n)
    return pd.DataFrame({
        "session_id": [f"S{i:07d}" for i in range(n)],
        TARGET: energy.clip(1, None).round(2),
        "hour": hour,
        "dow": dow,
        "charger_max_power_kw": power,
        "connector": connector,
        "start_ts": start_ts,
    })


# Adapta tabla y columnas a tu capa gold si USE_SYNTHETIC = False
ATHENA_SQL = """
SELECT
  session_id,
  energy_kwh,
  hour(start_ts)          AS hour,
  day_of_week(start_ts)   AS dow,
  charger_max_power_kw,
  CASE connector_type WHEN 'CCS2' THEN 0 WHEN 'CHADEMO' THEN 1 ELSE 2 END AS connector,
  start_ts
FROM charging_sessions
WHERE start_ts >= date '2025-10-01' AND energy_kwh > 0
"""

df = synthetic_sessions() if USE_SYNTHETIC else wr.athena.read_sql_query(ATHENA_SQL, database=ATHENA_DATABASE)
df = df.sort_values("start_ts").reset_index(drop=True)
df.head()

In [ ]:
# Split TEMPORAL: pasado para train, futuro para validation y test (evita fuga de información)
n = len(df)
train, val, test = df.iloc[: int(n * 0.8)], df.iloc[int(n * 0.8): int(n * 0.9)], df.iloc[int(n * 0.9):]
for name, part in (("train", train), ("validation", val), ("test", test)):
    print(f"{name:<10} {len(part):>6} filas  {part.start_ts.min()} → {part.start_ts.max()}")

COLS = [TARGET] + FEATURES
base = f"s3://{BUCKET}/{DATASET}"

# CSV para XGBoost built-in: target en la primera columna y SIN cabecera
wr.s3.to_csv(train[COLS], f"{base}/csv/train/train.csv", index=False, header=False)
wr.s3.to_csv(val[COLS], f"{base}/csv/validation/validation.csv", index=False, header=False)
# Test con session_id y target delante para unir y evaluar en Batch Transform (Lab 5)
wr.s3.to_csv(test[["session_id"] + COLS], f"{base}/csv/test/test.csv", index=False, header=False)

# Parquet para script mode (Lab 3)
wr.s3.to_parquet(train[COLS], f"{base}/parquet/train/part-0.parquet")
wr.s3.to_parquet(val[COLS], f"{base}/parquet/validation/part-0.parquet")

wr.s3.list_objects(f"{base}/")

## Lab 2 · Training job con boto3 (XGBoost built-in)

Objetivo: ver cada pieza de `CreateTrainingJob`: imagen, canales, salida, recursos, Spot y checkpoints.
Mientras corre, mira en la consola los *secondary statuses* y los logs en CloudWatch (`/aws/sagemaker/TrainingJobs`).

In [ ]:
xgb_image = image_uris.retrieve(framework="xgboost", region=REGION, version="1.7-1")
xgb_job_name = f"kwh-xgb-{TS}"


def csv_channel(name: str) -> dict:
    return {
        "ChannelName": name,
        "DataSource": {"S3DataSource": {
            "S3DataType": "S3Prefix",
            "S3Uri": f"s3://{BUCKET}/{DATASET}/csv/{name}/",
            "S3DataDistributionType": "FullyReplicated",
        }},
        "ContentType": "text/csv",
    }


sm.create_training_job(
    TrainingJobName=xgb_job_name,
    RoleArn=ROLE_ARN,
    AlgorithmSpecification={"TrainingImage": xgb_image, "TrainingInputMode": "File"},
    HyperParameters={  # en la API siempre son strings
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "num_round": "300",
        "max_depth": "6",
        "eta": "0.1",
        "early_stopping_rounds": "20",
    },
    InputDataConfig=[csv_channel("train"), csv_channel("validation")],
    OutputDataConfig={"S3OutputPath": f"s3://{BUCKET}/{BASE}/models/"},
    ResourceConfig={"InstanceType": "ml.m5.xlarge", "InstanceCount": 1, "VolumeSizeInGB": 10},
    EnableManagedSpotTraining=True,
    StoppingCondition={"MaxRuntimeInSeconds": 3600, "MaxWaitTimeInSeconds": 7200},
    CheckpointConfig={"S3Uri": f"s3://{BUCKET}/{BASE}/checkpoints/{xgb_job_name}/"},
)
print("Lanzado:", xgb_job_name)
sm.get_waiter("training_job_completed_or_stopped").wait(
    TrainingJobName=xgb_job_name, WaiterConfig={"Delay": 30, "MaxAttempts": 240}
)

In [ ]:
desc = sm.describe_training_job(TrainingJobName=xgb_job_name)
print("Estado:", desc["TrainingJobStatus"], desc.get("FailureReason", ""))
print("Métricas finales:", desc.get("FinalMetricDataList"))
print(f"Spot: facturados {desc.get('BillableTimeInSeconds')} s de {desc.get('TrainingTimeInSeconds')} s")
MODEL_DATA = desc["ModelArtifacts"]["S3ModelArtifacts"]
print("Artefacto:", MODEL_DATA)

## Lab 3 · Script mode con SDK v3 y FastFile

Objetivo: tu propio código (`src/train.py`), datos en Parquet y modo FastFile.

Extra: con Docker instalado puedes iterar sin coste con `Compute(instance_type="local_cpu")`
y `training_mode=Mode.LOCAL_CONTAINER` (`from sagemaker.train.model_trainer import Mode`).

In [ ]:
from sagemaker.core.helper.session_helper import Session
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, InputData, OutputDataConfig, SourceCode

session = Session(boto_session=boto3.Session(region_name=REGION))
sklearn_image = image_uris.retrieve(
    framework="sklearn", region=REGION, version="1.2-1",
    py_version="py3", instance_type="ml.m5.xlarge", image_scope="training",
)

trainer = ModelTrainer(
    training_image=sklearn_image,
    role=ROLE_ARN,
    sagemaker_session=session,
    source_code=SourceCode(source_dir="./src", entry_script="train.py", requirements="requirements.txt"),
    compute=Compute(instance_type="ml.m5.xlarge", instance_count=1),
    hyperparameters={"max_iter": 300, "learning_rate": 0.05},
    training_input_mode="FastFile",
    output_data_config=OutputDataConfig(s3_output_path=f"s3://{BUCKET}/{BASE}/models/"),
    base_job_name="kwh-sklearn",
)

trainer.train(input_data_config=[
    InputData(channel_name="train", data_source=f"s3://{BUCKET}/{DATASET}/parquet/train/"),
    InputData(channel_name="validation", data_source=f"s3://{BUCKET}/{DATASET}/parquet/validation/"),
])

In [ ]:
# El job aparece igual que el del Lab 2: el SDK solo es una capa sobre CreateTrainingJob
last = sm.list_training_jobs(
    NameContains="kwh-sklearn", SortBy="CreationTime", SortOrder="Descending", MaxResults=1
)["TrainingJobSummaries"][0]
print(last["TrainingJobName"], last["TrainingJobStatus"])
# Busca la línea "val_mae=..." en CloudWatch y compárala con el RMSE del XGBoost

## Lab 4 · Endpoint real-time + data capture + autoscaling

Objetivo: entender la cadena **Model → EndpointConfig → Endpoint**.

Reto: entrena otra versión, crea un EndpointConfig nuevo y lanza `sm.update_endpoint(...)`
mientras invocas en bucle. No debería haber errores (blue/green por debajo).

In [ ]:
model_name = f"kwh-xgb-{TS}"
sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=ROLE_ARN,
    PrimaryContainer={"Image": xgb_image, "ModelDataUrl": MODEL_DATA},
)

sm.create_endpoint_config(
    EndpointConfigName=f"{model_name}-rt",
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InstanceType": "ml.m5.large",
        "InitialInstanceCount": 1,
    }],
    DataCaptureConfig={  # base para Model Monitor
        "EnableCapture": True,
        "InitialSamplingPercentage": 100,
        "DestinationS3Uri": f"s3://{BUCKET}/{BASE}/datacapture/",
        "CaptureOptions": [{"CaptureMode": "Input"}, {"CaptureMode": "Output"}],
    },
)
sm.create_endpoint(EndpointName="kwh-rt", EndpointConfigName=f"{model_name}-rt")
sm.get_waiter("endpoint_in_service").wait(EndpointName="kwh-rt")
print("Endpoint kwh-rt InService")

In [ ]:
# hour, dow, charger_max_power_kw, connector (sin target)
resp = smr.invoke_endpoint(EndpointName="kwh-rt", ContentType="text/csv", Body="18,4,150,0\n8,1,50,1")
print(resp["Body"].read().decode())

In [ ]:
# Autoscaling por invocaciones por instancia (opcional)
RT_RESOURCE_ID = "endpoint/kwh-rt/variant/AllTraffic"
DIMENSION = "sagemaker:variant:DesiredInstanceCount"

aas.register_scalable_target(
    ServiceNamespace="sagemaker", ResourceId=RT_RESOURCE_ID,
    ScalableDimension=DIMENSION, MinCapacity=1, MaxCapacity=3,
)
aas.put_scaling_policy(
    PolicyName="kwh-invocations", ServiceNamespace="sagemaker", ResourceId=RT_RESOURCE_ID,
    ScalableDimension=DIMENSION, PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 100.0,
        "PredefinedMetricSpecification": {"PredefinedMetricType": "SageMakerVariantInvocationsPerInstance"},
        "ScaleOutCooldown": 60,
        "ScaleInCooldown": 300,
    },
)

## Lab 5 · Batch Transform con unión de resultados

Objetivo: puntuar un fichero entero sin endpoint y obtener `session_id, real, predicho`.
Los filtros usan JSONPath sobre las columnas del CSV.

In [ ]:
bt_name = f"kwh-bt-{TS}"
bt_output = f"s3://{BUCKET}/{BASE}/batch-out/{bt_name}/"

sm.create_transform_job(
    TransformJobName=bt_name,
    ModelName=model_name,  # el mismo Model del Lab 4
    TransformInput={
        "DataSource": {"S3DataSource": {
            "S3DataType": "S3Prefix",
            "S3Uri": f"s3://{BUCKET}/{DATASET}/csv/test/",
        }},
        "ContentType": "text/csv",
        "SplitType": "Line",
    },
    TransformOutput={"S3OutputPath": bt_output, "AssembleWith": "Line", "Accept": "text/csv"},
    TransformResources={"InstanceType": "ml.m5.large", "InstanceCount": 1},
    BatchStrategy="MultiRecord",
    MaxPayloadInMB=6,
    DataProcessing={
        "InputFilter": "$[2:]",       # al modelo solo le llegan las features
        "JoinSource": "Input",        # pega la predicción a la fila original
        "OutputFilter": "$[0,1,-1]",  # session_id, energy_kwh real, predicción
    },
)
sm.get_waiter("transform_job_completed_or_stopped").wait(TransformJobName=bt_name)
print(sm.describe_transform_job(TransformJobName=bt_name)["TransformJobStatus"])

In [ ]:
scored = wr.s3.read_csv(bt_output, header=None, names=["session_id", "real", "pred"])
print(f"MAE en test: {(scored.real - scored.pred).abs().mean():.3f} kWh")
scored.head()

## Lab 6 · Serverless y asíncrona con el mismo Model

Objetivo: comparar cold start en serverless y el patrón petición en S3 → respuesta en S3 de la asíncrona.

In [ ]:
sm.create_endpoint_config(
    EndpointConfigName=f"{model_name}-sls",
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "ServerlessConfig": {"MemorySizeInMB": 2048, "MaxConcurrency": 5},
    }],
)
sm.create_endpoint(EndpointName="kwh-sls", EndpointConfigName=f"{model_name}-sls")
sm.get_waiter("endpoint_in_service").wait(EndpointName="kwh-sls")

for intento in ("primera (fría)", "segunda (caliente)"):
    t0 = time.perf_counter()
    smr.invoke_endpoint(EndpointName="kwh-sls", ContentType="text/csv", Body="18,4,150,0")
    print(f"{intento}: {time.perf_counter() - t0:.2f} s")

In [ ]:
sm.get_waiter("endpoint_in_service").wait(EndpointName="kwh-sls")  # ya está InService: vuelve al instante

for intento in ("primera (fría)", "segunda (caliente)"):
    t0 = time.perf_counter()
    smr.invoke_endpoint(EndpointName="kwh-sls", ContentType="text/csv", Body="18,4,150,0")
    print(f"{intento}: {time.perf_counter() - t0:.2f} s")

In [ ]:
sm.create_endpoint_config(
    EndpointConfigName=f"{model_name}-async",
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InstanceType": "ml.m5.large",
        "InitialInstanceCount": 1,
    }],
    AsyncInferenceConfig={"OutputConfig": {"S3OutputPath": f"s3://{BUCKET}/{BASE}/async-out/"}},
)
sm.create_endpoint(EndpointName="kwh-async", EndpointConfigName=f"{model_name}-async")
sm.get_waiter("endpoint_in_service").wait(EndpointName="kwh-async")

# La petición vive en S3, no en el body
req_key = f"{BASE}/async-in/req-{TS}.csv"
s3.put_object(Bucket=BUCKET, Key=req_key, Body=b"18,4,150,0\n8,1,50,1")
r = smr.invoke_endpoint_async(
    EndpointName="kwh-async", ContentType="text/csv", InputLocation=f"s3://{BUCKET}/{req_key}"
)
out_bucket, out_key = r["OutputLocation"].removeprefix("s3://").split("/", 1)
print("Resultado irá a:", r["OutputLocation"])

for _ in range(30):
    try:
        print(s3.get_object(Bucket=out_bucket, Key=out_key)["Body"].read().decode())
        break
    except s3.exceptions.NoSuchKey:
        time.sleep(5)
else:
    print("Todavía no hay resultado; revisa async-out/ más tarde o los logs del endpoint")

# Reto: autoscaling del endpoint asíncrono con MinCapacity=0 y una política sobre la métrica
# HasBacklogWithoutCapacity para que arranque desde cero cuando llegue trabajo.

## Lab 7 · Limpieza (no te lo saltes)

In [ ]:
try:
    aas.deregister_scalable_target(
        ServiceNamespace="sagemaker", ResourceId=RT_RESOURCE_ID, ScalableDimension=DIMENSION
    )
except (ClientError, NameError) as e:
    print("Autoscaling:", e)

for ep in ("kwh-rt", "kwh-sls", "kwh-async"):
    try:
        sm.delete_endpoint(EndpointName=ep)
        print("Borrado endpoint", ep)
    except ClientError as e:
        print(ep, "→", e.response["Error"]["Message"])

for cfg in sm.list_endpoint_configs(NameContains="kwh-xgb")["EndpointConfigs"]:
    sm.delete_endpoint_config(EndpointConfigName=cfg["EndpointConfigName"])
    print("Borrado endpoint config", cfg["EndpointConfigName"])

for m in sm.list_models(NameContains="kwh-xgb")["Models"]:
    sm.delete_model(ModelName=m["ModelName"])
    print("Borrado model", m["ModelName"])

In [ ]:
# Comprobación final: solo deberían quedar endpoints en estado Deleting (desaparecen en unos minutos)
print([(e["EndpointName"], e["EndpointStatus"]) for e in sm.list_endpoints(NameContains="kwh-")["Endpoints"]])
# Los datos de S3 cuestan poco; bórralos si no vas a repetir los labs:
# wr.s3.delete_objects(f"s3://{BUCKET}/{BASE}/")